# ДЗ 3. NumPy на данных «Дала Маркет»
Срок: **до четверга 12.11, 19:00**.
Сдача — папка `hw03` репозитория `dsa-homework` (`hw03.ipynb`): `git add hw03` → `git commit` → `git push`.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем — только массивы и маски (кроме задачи 8).

- **[База]** обязательно для всех · **[Стандарт]** — сколько получится · **[Челлендж]** — для тех, кому легко.

## [Для всех] Чтение кода — без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
# 1
a = np.arange(6).reshape(2, 3); print(a[1, 2], a.shape)
# 2
a = np.array([3, 8, 1, 8]); print(a.argmax(), a.max())
# 3
a = np.arange(10); b = a[::3]; b[0] = 99; print(a[0])
# 4
a = np.array([1, 5, 7, 2]); print((a > 3).sum(), (a > 3).mean())
# 5
print(np.zeros((2, 3)).shape, np.linspace(0, 1, 5)[1])
```

Мои предсказания и объяснения:

1. 
2. 
3. 
4. 
5. 

In [ ]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

In [ ]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

# функции из урока 2 + превращаем колонки в массивы NumPy
amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])

print(amount.shape, amount.dtype, paid.dtype)

## [База]
1. Найдите позицию (`argmax`) самого дорогого заказа. Какая сумма, клиент и город?
2. Выручка оплаченных заказов и сумма неоплаченных — через маску `paid` и `~paid`.
3. Сколько заказов в Шымкенте и какая у них доля от всех заказов?

In [ ]:
#1
max_position = int(amount.argmax())
print(f"1. Индекс {max_position}; сумма {amount[max_position]:,.2f} ₸; "
    f"клиент {cust[max_position]}; город {city[max_position]}.")
print("Менеджеру: самый дорогой заказ относится к TEST; его нужно проверить как тестовую операцию.")

#2
paid_revenue = amount[paid].sum()
unpaid_total = amount[~paid].sum()
print(f"2. Выручка: {paid_revenue:,.2f} ₸; неоплачено: {unpaid_total:,.2f} ₸.")
print("Менеджеру: неоплаченные суммы не включаем в выручку.")

#3
shymkent = city == "Шымкент"
print(f"3. Шымкент: {shymkent.sum()} заказов, {shymkent.mean():.2%} от всех.")
print("Менеджеру: эта доля показывает вклад Шымкента в число заказов, а не в выручку.")

## [Стандарт]
4. Маркетинг предлагает скидку 15% на оплаченные заказы дороже 50 000 ₸. Сколько выручки магазин недополучит?
   Посчитайте дважды: со всеми заказами и без заказов клиента TEST. Почему ответы различаются в несколько раз?
5. Для **каждого** города — доля заказов, доставленных за 1–2 дня. Список городов: `np.unique(city)`; цикл по 6 городам здесь можно.
6. Ниже код, который AI написал на запрос «посчитай средний чек». Найдите две ошибки, исправьте и посчитайте правильно:
```python
avg = amount.mean()
print("Средний чек:", round(avg))
```

In [ ]:
# 4. Скидка только на оплаченные заказы строго дороже 50 000.
non_test = cust != "TEST"
discount_mask = paid & (amount > 50_000)
loss_all = amount[discount_mask].sum() * 0.15
loss_real = amount[discount_mask & non_test].sum() * 0.15
print(f"4. Потери со всеми заказами: {loss_all:,.2f} ₸; без TEST: {loss_real:,.2f} ₸.")
print(f"Менеджеру: TEST завышает оценку потерь в {loss_all / loss_real:.2f} раза, "
      f"добавляя {loss_all - loss_real:,.2f} ₸ скидок на тестовые суммы.")

# 5. Знаменатель — все заказы города, как указано в условии.
# NaN не входит в 1–2 дня: неизвестный срок не означает доказанную задержку.
fast = (days >= 1) & (days <= 2)
print("5. Доля доставки за 1–2 дня среди всех заказов города:")
for current_city in np.unique(city):
    city_mask = city == current_city
    fast_count = (city_mask & fast).sum()
    city_count = city_mask.sum()
    missing_count = np.isnan(days[city_mask]).sum()
    print(f"{current_city}: {fast_count}/{city_count} = {fast_count / city_count:.2%}; "
        f"неизвестный срок: {missing_count}.")
    print(f"Менеджеру: в городе {current_city} подтверждена быстрая доставка "
        f"для {fast_count / city_count:.2%} всех заказов.")

# 6. Две ошибки: учтены неоплаченные заказы и тестовый клиент.
# Повторы здесь не удаляем: отдельного требования об этом в ДЗ 3 нет.
avg = amount[paid & non_test].mean()
print(f"6. Средний чек оплаченных заказов без TEST: {avg:,.2f} ₸.")
print("Менеджеру: этот средний чек исключает неоплаченные суммы и искусственно большие заказы TEST.")

## [Челлендж]
7. Сравните скорость: сумма оплаченных заказов циклом `for` и через маску. Во сколько раз быстрее?
8. Смоделируйте A/B-тест. `np.random.seed(42)`. Группа A: 1 000 клиентов, вероятность покупки 5%; группа B: 1 000 клиентов, 6%
   (покупка = `np.random.rand(1000) < 0.05`). Сколько покупок в каждой группе?
   Повторите эксперимент 100 раз: в скольких случаях B оказалась лучше A? Что это говорит о размере выборки? (2–3 предложения)

In [ ]:
# 7. Цикл по заказам нужен только для сравнения, запрошенного в задаче 7.
from timeit import repeat

def sum_loop():
    total = 0.0
    for value, is_paid in zip(amount, paid):
        if is_paid:
            total += value
    return total

def sum_numpy():
    return amount[paid].sum()

assert np.isclose(sum_loop(), sum_numpy())
# Одинаковые данные и число вызовов; маска применяется внутри замера NumPy.
repeats = 200
loop_seconds = min(repeat(sum_loop, number=repeats, repeat=5)) / repeats
numpy_seconds = min(repeat(sum_numpy, number=repeats, repeat=5)) / repeats
speedup = loop_seconds / numpy_seconds
print(f"7. Цикл: {loop_seconds * 1e6:.2f} мкс; NumPy: {numpy_seconds * 1e6:.2f} мкс; "
      f"отношение времени цикла к NumPy: {speedup:.2f}.")
print("Менеджеру: обе версии дают одинаковую выручку; измеренная скорость зависит от компьютера и размера данных.")

# 8. Один эксперимент, затем воспроизводимая серия из 100.
np.random.seed(42)
purchases_a = (np.random.rand(1000) < 0.05).sum()
purchases_b = (np.random.rand(1000) < 0.06).sum()
print(f"8. Первый эксперимент: A — {purchases_a} покупок, B — {purchases_b}.")

# Повторный seed означает, что первый эксперимент входит в серию из 100.
np.random.seed(42)
wins_b = 0
ties = 0
for experiment in range(100):
    count_a = (np.random.rand(1000) < 0.05).sum()
    count_b = (np.random.rand(1000) < 0.06).sum()
    wins_b += int(count_b > count_a)
    ties += int(count_b == count_a)
print(f"Из 100 экспериментов: B лучше A — {wins_b}; ничья — {ties}; A лучше B — {100 - wins_b - ties}.")
print(f"Менеджеру: B победила в {wins_b} из 100 экспериментов, хотя её вероятность покупки выше во всех опытах. "
      "При 1 000 клиентах в группе случайные колебания могут скрывать разницу в 1 процентный пункт. "
      "Большая выборка уменьшит случайный разброс; само число побед не является тестом статистической значимости.")